# Compare reported fields and model estimates

Read the original facility fields, linked contacts and branch IDs. Save a sidecar
`field_updates` table with one provenance-tagged value per field, facility and method.
Later notebooks display model estimates beside the evidence while keeping the
original data and the service claims available for inspection.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Observed counts, contacts and optional imported estimates

An unambiguous exact count is copied from the original FDR field with the
quoted value and source record ID. Ranges and contradictory counts remain in
the audit from notebook 03. Linked phone and email evidence stays traceable
to the source listing. Multiple published contacts can coexist.

Set `estimate_file` to an already-reviewed CSV or JSONL with `facility_id`,
`field`, `method`, `value`, `unit`, `lower`, `upper`, `provenance` and `detail`.
Method must be `ml` or `regression`. Supply the model name, version, evaluation
cohort and interval definition in the provenance and detail where available.
The demo uses hand-written overlay values so each mode can be compared locally.

In [ ]:
from facility_tool.eda import COUNT_UNITS, reported_count_updates
from facility_tool.examples import demo_overlays, demo_pressure
from facility_tool.updates import contact_evidence, empty_updates, load_estimates

entities = STORE.read("entities")
records = STORE.read("records")
links = STORE.read("links")
linked = STORE.read("linked_fdr")
estimate_file = None  # Optional path to existing results for this snapshot.
observed = [reported_count_updates(linked, field) for field in COUNT_UNITS if field in linked.columns]
imported = load_estimates(estimate_file, entities["facility_id"], services=config["service_aliases"])
contacts = contact_evidence(records, links)
STORE.write("contact_evidence", contacts)

## Keep service load data separate

Bed counts and doctor counts are resource indicators. The separate load
tables require the relevant service and unit. Demo availability is dated
fiction. Demo pressure is an illustrative demand/capacity scenario. Real
snapshots start with empty load tables until eligible observations are supplied.

In [ ]:
measured_load = pd.DataFrame(columns=["facility_id", "service", "unit", "available_units", "as_of", "source_url"])
pressure_scenarios = pd.DataFrame(columns=["facility_id", "service", "unit", "period", "basis", "demand", "capacity", "source_url"])
if MODE == "demo":
    fixture_estimates, measured_load = demo_overlays(entities, config["as_of"])
    pressure_scenarios = demo_pressure(entities)
    if not estimate_file:
        imported = fixture_estimates

frames = [frame for frame in [*observed, imported] if not frame.empty]
updates = pd.concat(frames, ignore_index=True) if frames else empty_updates()
STORE.write("field_updates", updates)
STORE.write("measured_load", measured_load)
STORE.write("pressure_scenarios", pressure_scenarios)
show(updates.groupby(["method", "field"]).size().rename("facility_values").reset_index())
show(updates.loc[updates["method"].isin(["ml", "regression"])], limit=12)
print(f"Cited contact rows: {len(contacts):,} | Dated load rows: {len(measured_load):,} | Pressure scenarios: {len(pressure_scenarios):,}")